# 第 9 章：SFT 指令微调

这个 notebook 对应 `lessons/09_sft_instruction_tuning.md`，演示 SFT 样本 schema、chat template、assistant-only label mask、source group 切分，以及训练前后行为对比报告。

In [ ]:
import tempfile
from pathlib import Path

from src.data.text_datasets import ChatMessage
from src.finetune.sft import (
    SFTExample,
    build_sft_batch_item,
    compare_behaviors,
    dump_sft_jsonl,
    load_sft_jsonl,
    render_training_text,
    split_by_source_group,
    supervised_label_text,
    write_behavior_report,
)
from src.tokenizer.simple_tokenizer import CharacterTokenizer

## 1. SFT Example Schema

SFT 样本保持结构化，至少要有 id、messages、source、source_group 和 assistant 回答。

In [ ]:
example = SFTExample(
    id="sft_001",
    source="manual",
    source_group="doc_a",
    risk_tags=["teaching"],
    messages=[
        ChatMessage(role="system", content="你是技术助教"),
        ChatMessage(role="user", content="解释 causal mask"),
        ChatMessage(role="assistant", content="只能看历史 token"),
    ],
)
print(example)

## 2. Chat Template Text

训练和推理必须使用同一模板。没有模型自带模板时，项目需要固定 fallback 格式。

In [ ]:
training_text = render_training_text(example)
print(training_text)

## 3. Assistant-Only Label Mask

system/user/padding 不参与 loss，监督文本只应包含 assistant 回答。

In [ ]:
tokenizer = CharacterTokenizer.from_texts([training_text])
features = build_sft_batch_item(example, tokenizer, max_length=80)
print(features.input_ids.shape)
print(features.labels)
print("supervised:", supervised_label_text(features, tokenizer))

## 4. JSONL 与 Source Group Split

切分不能让同一个 source group 同时出现在 train 和 val/test。

In [ ]:
examples = [
    example,
    SFTExample(
        id="sft_002",
        source="manual",
        source_group="doc_b",
        messages=[
            ChatMessage(role="user", content="解释 embedding"),
            ChatMessage(role="assistant", content="embedding 是可训练查表矩阵"),
        ],
    ),
    SFTExample(
        id="sft_003",
        source="manual",
        source_group="doc_c",
        messages=[
            ChatMessage(role="user", content="解释 dropout"),
            ChatMessage(role="assistant", content="dropout 在训练时随机丢弃激活"),
        ],
    ),
]

with tempfile.TemporaryDirectory() as tmpdir:
    path = Path(tmpdir) / "sft.jsonl"
    dump_sft_jsonl(path, examples)
    loaded = load_sft_jsonl(path)
    train, val = split_by_source_group(loaded, val_ratio=0.34, seed=0)

print([item.id for item in train], [item.id for item in val])

## 5. Before / After 行为对比

SFT 报告应比较同一 prompt 在训练前后的输出变化，而不是只看 loss。

In [ ]:
comparisons = compare_behaviors(
    prompts=["解释 causal mask"],
    before_outputs=["继续写 causal mask"],
    after_outputs=["causal mask 只能看历史 token"],
)

with tempfile.TemporaryDirectory() as tmpdir:
    report_path = Path(tmpdir) / "before_after.md"
    write_behavior_report(report_path, comparisons)
    print(report_path.read_text())